# ID Forensics — Colab Workbench

Run only the sections you need. Each section is independent.

**Persistent on Drive** (`My Drive/id-forensics/`): images, weights, eval reports

**Colab Secrets** (🔑 sidebar):
- S3 / training / OCR audit: `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION`, `AWS_SESSION_TOKEN`
- Batch labeling (section 7, optional DB): `ZENKA_KE_DATABASE_URL`

**Section 8 (OCR audit)** needs the latest GitHub code (`export_ocr_audit.py`, `field_localization/`). Run section 1 `git pull` after pushing from your PC.

## Config

In [ ]:
GITHUB_TOKEN = ""   # only if repo is private

# --- Data pipeline toggles ---
# Set SYNC_IMAGES=True the first time, or after adding new labeled images from S3.
# Stage 2 and Stage 3 read images from Drive; Stage 1 downloads quality-gate images from S3.
SYNC_IMAGES = False     # True: download new images S3→Drive (needed for stage2 + stage3)
REBUILD_DATASET = True  # True: convert labels → training splits

# --- Label Studio export ---
# Upload your Label Studio JSON export to Drive at this path.
LS_EXPORT_PATH = "/content/drive/MyDrive/id-forensics/label_studio_export.json"

# --- Training — list stages to run; others are skipped ---
# Options: "stage1", "stage2", "stage3"
# stage4 (field localization/OCR) has no ML training yet
# To retrain everything: ["stage1", "stage2", "stage3"]
TRAIN_STAGES = ["stage1"]

RUN_EVAL = True
EVAL_SPLIT = "val"

# Hyperparameters
DEVICE = "cuda"
STAGE1_EPOCHS, STAGE1_BATCH = 40, 32
STAGE2_EPOCHS, STAGE2_BATCH, STAGE2_MODEL = 100, 16, "yolov8s-pose.pt"
STAGE3_EPOCHS, STAGE3_BATCH = 50, 32

---
## 1. Connect — Drive + GitHub + deps

Mounts Drive, pulls latest code + labels from GitHub, installs Python packages.
Run every session.

In [ ]:
import os, sys, importlib, subprocess
from pathlib import Path

import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

REPO = Path("/content/id-forensics-model")
user, repo = "ansisvaisla", "id-forensics-model"
url = (
    f"https://{GITHUB_TOKEN}@github.com/{user}/{repo}.git"
    if GITHUB_TOKEN else f"https://github.com/{user}/{repo}.git"
)

os.chdir("/content")
if not REPO.is_dir():
    subprocess.run(["git", "clone", url, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "origin", "main"], check=False)

sys.path.insert(0, str(REPO / "scripts"))
import colab_bootstrap as cb
importlib.reload(cb)

cb.connect(github_token=GITHUB_TOKEN)

---
## 2. Sync images from S3 → Drive

**When to run:** first time, or after adding new labeled images.

In [ ]:
if SYNC_IMAGES:
    # Copy the LS export into the repo NOW so sync_images_from_s3 reads the correct file
    from pathlib import Path as _P
    import shutil as _sh
    _ls = _P(LS_EXPORT_PATH)
    if _ls.is_file():
        _dest = _P("/content/id-forensics-model/data/labels/label_studio_export.json")
        _dest.parent.mkdir(parents=True, exist_ok=True)
        _sh.copy2(str(_ls), str(_dest))
        print(f"Copied LS export → {_dest}")
    else:
        print(f"WARNING: LS export not found at {LS_EXPORT_PATH} — sync will use old export")
    cb.sync_images_from_s3()
else:
    n = len(list(cb.DRIVE_RAW_DIR.glob("*.jpg")))
    print(f"Skipped S3 sync — {n} images already on Drive")

---
## 3. Build training datasets

**When to run:** after pushing new labels to GitHub, or after syncing new images.

In [ ]:
if REBUILD_DATASET:
    from pathlib import Path as _P
    import os as _os
    _ls = _P(LS_EXPORT_PATH)

    if not _ls.is_file():
        print(f"WARNING: LS export not found at {LS_EXPORT_PATH}")
        print("Upload your Label Studio JSON export to Drive at the path above.")
    else:
        import shutil as _sh
        _dest = _P("/content/id-forensics-model/data/labels/label_studio_export.json")
        _dest.parent.mkdir(parents=True, exist_ok=True)
        _sh.copy2(str(_ls), str(_dest))
        print(f"Copied LS export → {_dest}")

        # Inject AWS credentials so quality gate script can download from S3
        from google.colab import userdata as _ud
        for _s in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_DEFAULT_REGION", "AWS_SESSION_TOKEN"]:
            _v = _ud.get(_s) or ""
            if _v:
                _os.environ[_s] = _v

        # Stage 1 (quality gate): build 8-class dataset.
        # Images are cached on Drive — only downloaded from S3 on first run.
        # Must run BEFORE rebuild_splits so it is not overwritten by split_yolo_dataset.
        if "stage1" in TRAIN_STAGES:
            print("\nBuilding quality gate dataset ...")
            !cd /content/id-forensics-model && python scripts/convert_labels_ls_to_quality_gate.py \
                --input {LS_EXPORT_PATH} --out data/yolo/screen --val-ratio 0.2 \
                --image-cache {cb.DRIVE_SCREEN_DIR}

        # Stage 2 + Stage 3: read images from Drive.
        # skip_screen=True prevents split_yolo_dataset from overwriting the 8-class quality gate data.
        cb.rebuild_splits(skip_screen="stage1" in TRAIN_STAGES)
else:
    print("Skipped dataset rebuild")

---
## 4. Train models

Only stages listed in `TRAIN_STAGES` run. Others print "Skipping".

### Stage 1 — Quality Gate (8-class: screen / printout / selfie / back / garbage / good_front / partial / blurry)

In [ ]:
if "stage1" in TRAIN_STAGES:
    !cd /content/id-forensics-model && python scripts/training/train_stage2_screen.py \
        --device {DEVICE} --epochs {STAGE1_EPOCHS} --batch {STAGE1_BATCH}
else:
    print("Skipping Stage 1")

### Stage 2 — corner detection (YOLO Pose)

In [ ]:
if "stage2" in TRAIN_STAGES:
    cb.clear_corner_caches()
    !cd /content/id-forensics-model && python scripts/training/train_stage1_corners.py \
        --model {STAGE2_MODEL} --device 0 --epochs {STAGE2_EPOCHS} --batch {STAGE2_BATCH}
else:
    print("Skipping Stage 2")

### Stage 3 — ID type classifier (EfficientNet-B0)

Requires Stage 2 corner weights for cropping/deskewing. Load from Drive if not trained this session.

In [ ]:
if "stage3" in TRAIN_STAGES:
    cb.restore_weights("stage2")
    !cd /content/id-forensics-model && python scripts/deskew_id_type_images.py
    !cd /content/id-forensics-model && python scripts/split_id_type_dataset.py --source all_deskewed
    !cd /content/id-forensics-model && python scripts/training/train_stage4_id_type.py \
        --device {DEVICE} --epochs {STAGE3_EPOCHS} --batch {STAGE3_BATCH}
else:
    print("Skipping Stage 3")

### Stage 4 — field localization and OCR

No ML training yet. This is template-based and runs in the pipeline after Stage 3.

In [ ]:
if "stage4" in TRAIN_STAGES:
    print("Stage 4 has no ML training yet — field localization/OCR is template-based.")
    print("Remove 'stage4' from TRAIN_STAGES unless you are running OCR audit scripts manually.")
else:
    print("Skipping Stage 4")

---
## 5. Save weights to Drive

In [ ]:
for stage in TRAIN_STAGES:
    if stage == "stage4":
        continue
    try:
        cb.save_weights(stage)
    except FileNotFoundError as exc:
        print(f"Could not save {stage}: {exc}")

---
## 6. Evaluate (saved to Drive)

In [ ]:
if RUN_EVAL:
    for stage in TRAIN_STAGES:
        if stage == "stage4":
            print("Skipping Stage 4 eval (template OCR uses a separate field-level audit)")
            continue
        try:
            eval_dir = cb.run_eval(stage, split=EVAL_SPLIT)
            print(f"  wrong images: {eval_dir / 'viz' / 'wrong'}")
        except Exception as exc:
            print(f"Eval failed for {stage}: {exc}")
else:
    print("Skipping eval")

---
## 7. Batch Labeling → Label Studio

Generate a pre-annotated Label Studio import JSON from recent DB images.

**Colab Secrets required** (🔑 sidebar — in addition to AWS ones):

| Secret | Value |
|--------|-------|
| `ZENKA_KE_DATABASE_URL` | `postgresql://<user>:<secret>@<host>:5432/<dbname>` *(recommended)* |
| *or individually* | `ZENKA_KE_DB_HOST`, `ZENKA_KE_DB_USER`, `ZENKA_KE_DB_PASSWORD`, `ZENKA_KE_DB_NAME` |

Output is saved to **Drive** `My Drive/id-forensics/data/batches/`.
After importing + correcting in Label Studio → export → push → retrain.

### Batch config

**No DB needed** — uses `data/batches/candidates.csv` exported from DBeaver on work PC.
AWS secrets (🔑) are used for image download + presigned URLs.

In [ ]:
BATCH_LIMIT          = 1000   # max images per batch
BATCH_HOURS          = 720    # look-back window (default 720h = 30 days)
BATCH_SKIP_INFERENCE = False  # True = no pipeline, just tasks with no predictions (faster)
BATCH_URL_EXPIRY     = 604800 # presigned URL lifetime in seconds (7 days)

In [ ]:
import os
from pathlib import Path
from datetime import datetime
from google.colab import userdata

# Inject AWS secrets into env
for _s in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_DEFAULT_REGION", "AWS_SESSION_TOKEN"]:
    _v = userdata.get(_s) or ""
    if _v:
        os.environ[_s] = _v

# Skip Textract — not needed for batch labeling, avoids AccessDenied slowdown
os.environ["SKIP_FIELD_EXTRACTOR"] = "1"

# Build DB DSN from individual parts
_host = (userdata.get("ZENKA_KE_DB_HOST") or "").strip()
_user = (userdata.get("ZENKA_KE_DB_USER") or "").strip()
_pwd  = (userdata.get("ZENKA_KE_DB_PASSWORD") or "").strip()
_name = (userdata.get("ZENKA_KE_DB_NAME") or "").strip()
_port = (userdata.get("ZENKA_KE_DB_PORT") or "5432").strip()
if _host and _user and _pwd and _name:
    from urllib.parse import quote
    os.environ["ZENKA_KE_DATABASE_URL"] = f"postgresql://{quote(_user,safe='')}:{quote(_pwd,safe='')}@{_host}:{_port}/{_name}"

print("AWS_ACCESS_KEY_ID set:", bool(os.environ.get("AWS_ACCESS_KEY_ID")))
print("AWS_DEFAULT_REGION:", os.environ.get("AWS_DEFAULT_REGION", "NOT SET"))
print("SKIP_FIELD_EXTRACTOR:", os.environ.get("SKIP_FIELD_EXTRACTOR"))

out_dir  = cb.DRIVE_BATCHES_DIR
out_dir.mkdir(parents=True, exist_ok=True)
out_file = out_dir / f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_batch.json"

_skip = "--skip-inference" if BATCH_SKIP_INFERENCE else ""

In [ ]:
# Restore model weights from Drive (needed for pipeline inference)
if not BATCH_SKIP_INFERENCE:
    cb.restore_all_weights()

In [ ]:
!cd /content/id-forensics-model && python scripts/batch_label.py \
    --from-csv data/batches/candidates.csv \
    --limit {BATCH_LIMIT} \
    --s3-uris \
    --skip-field-extractor \
    --workers 32 \
    --out {out_file} \
    {_skip}

from pathlib import Path as _P
if _P(str(out_file)).is_file():
    sz = _P(str(out_file)).stat().st_size / 1024
    print(f"\nBatch JSON saved to Drive: {out_file}  ({sz:.0f} KB)")
    print("Label Studio → Import → select that file.")
    print("NOTE: Images use s3:// URIs — make sure S3 cloud storage is configured in LS.")
else:
    print("Output file not found — check errors above.")

---
## 8. Stage 4 — OCR field-localization audit

Export a CSV that joins Label Studio labels with stored AWS Rekognition OCR
(`aws_rekognition_detect_text.csv`). No new OCR API calls.

**Drive files needed:**
- `data/batches/aws_rekognition_detect_text.csv` — DBeaver export (~800 MB)
- Label Studio export JSON (full project export is OK — we filter to the 500-image OCR batch)

**Colab Secrets:** same AWS keys as section 7 (no DB needed when using `--ocr-csv`).

Output: `My Drive/id-forensics/eval/ocr_audit.csv`

### OCR audit config

In [ ]:
# Path to your full Label Studio export on Drive (3322 tasks is fine)
LS_FULL_EXPORT_PATH = "/content/drive/MyDrive/id-forensics/labels/project-1-at-2026-06-26-17-16-dafe0e23.json"

# Filtered 500-image OCR batch (written by the next cells)
LS_OCR_EXPORT_REPO = "/content/id-forensics-model/data/labels/label_studio_ocr_500.json"
LS_OCR_EXPORT_DRIVE = "/content/drive/MyDrive/id-forensics/labels/label_studio_ocr_500.json"

# DBeaver export of aws-rekognition-detect-text rows
OCR_CSV_PATH = "/content/drive/MyDrive/id-forensics/data/batches/aws_rekognition_detect_text.csv"

OCR_AUDIT_OUT = "/content/drive/MyDrive/id-forensics/eval/ocr_audit.csv"
OCR_AUDIT_LIMIT = 500
OCR_BATCH_CREATED_ON = "2026-06-26"  # import date of the 500-image OCR pilot batch

# True: filter full LS export → 500 OCR tasks. False: reuse existing filtered JSON.
FILTER_OCR_BATCH = True

### Credentials + preflight

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path
from google.colab import userdata

REPO = Path("/content/id-forensics-model")

# Pull latest code (export_ocr_audit.py + field_localization/ must be on main)
subprocess.run(["git", "-C", str(REPO), "pull", "origin", "main"], check=False)

# Inject AWS secrets (needed if any step downloads from S3)
for _s in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_DEFAULT_REGION", "AWS_SESSION_TOKEN"]:
    _v = userdata.get(_s) or ""
    if _v:
        os.environ[_s] = _v

print("AWS_ACCESS_KEY_ID set:", bool(os.environ.get("AWS_ACCESS_KEY_ID")))
print("AWS_DEFAULT_REGION:", os.environ.get("AWS_DEFAULT_REGION", "NOT SET"))

_required = [
    REPO / "scripts/export_ocr_audit.py",
    REPO / "scripts/filter_ocr_label_export.py",
    REPO / "field_localization/__init__.py",
]
_missing = [p for p in _required if not p.is_file()]
if _missing:
    raise FileNotFoundError(
        "OCR scripts missing after git pull. Push latest code from your PC, then re-run this cell.\n"
        + "\n".join(f"  - {p}" for p in _missing)
    )

_ocr_csv = Path(OCR_CSV_PATH)
if not _ocr_csv.is_file():
    raise FileNotFoundError(f"OCR CSV not on Drive: {OCR_CSV_PATH}")

_repo_csv = REPO / "data/batches/aws_rekognition_detect_text.csv"
_repo_csv.parent.mkdir(parents=True, exist_ok=True)
if not _repo_csv.is_file() or _repo_csv.stat().st_size != _ocr_csv.stat().st_size:
    print(f"Linking OCR CSV → {_repo_csv}")
    if _repo_csv.is_file() or _repo_csv.is_symlink():
        _repo_csv.unlink()
    _repo_csv.symlink_to(_ocr_csv)
else:
    print(f"OCR CSV already linked: {_repo_csv}")

Path(OCR_AUDIT_OUT).parent.mkdir(parents=True, exist_ok=True)
print("Preflight OK")

### Filter OCR batch (500 images from mixed export)

In [ ]:
import shutil
from pathlib import Path as _P

_ocr_export = _P(LS_OCR_EXPORT_REPO)

if FILTER_OCR_BATCH:
    _full = _P(LS_FULL_EXPORT_PATH)
    if not _full.is_file():
        raise FileNotFoundError(f"Full LS export not found: {LS_FULL_EXPORT_PATH}")
    !cd /content/id-forensics-model && python scripts/filter_ocr_label_export.py \
        --input {LS_FULL_EXPORT_PATH} \
        --created-on {OCR_BATCH_CREATED_ON} \
        --out {LS_OCR_EXPORT_REPO}
else:
    _drive = _P(LS_OCR_EXPORT_DRIVE)
    if _drive.is_file() and not _ocr_export.is_file():
        shutil.copy2(str(_drive), str(_ocr_export))
        print(f"Copied filtered export from Drive → {_ocr_export}")
    elif not _ocr_export.is_file():
        raise FileNotFoundError(
            "No filtered OCR export found. Set FILTER_OCR_BATCH=True or upload "
            + LS_OCR_EXPORT_DRIVE
        )
    else:
        print(f"Using existing filtered export: {_ocr_export}")

_drive_out = _P(LS_OCR_EXPORT_DRIVE)
_drive_out.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(str(_ocr_export), str(_drive_out))
print(f"Saved filtered export → {_drive_out}")

### Run OCR audit export

In [ ]:
!cd /content/id-forensics-model && python scripts/export_ocr_audit.py \
    --label-export {LS_OCR_EXPORT_REPO} \
    --ocr-csv data/batches/aws_rekognition_detect_text.csv \
    --limit {OCR_AUDIT_LIMIT} \
    --out {OCR_AUDIT_OUT}

from pathlib import Path as _P
import csv as _csv

_out = _P(OCR_AUDIT_OUT)
if _out.is_file():
    rows = list(_csv.DictReader(_out.open(encoding="utf-8")))
    found = sum(1 for r in rows if r.get("ocr_found") == "True")
    print(f"\nOCR audit CSV: {_out}")
    print(f"  rows: {len(rows)}  with OCR: {found}")
    print("Add expected_* columns in Sheets/Excel, then run evaluate_field_localization.py")
else:
    print("OCR audit CSV not created — check errors above.")